In [ ]:
class Greeter:
  def __init__(self, name):
    self.name = name

  def say_Hello(self):
    return f"Hello, {self.name}"

g = Greeter("Rafia")
g.say_Hello()

'Hello, Rafia'

In [31]:
import nltk
nltk.download('punkt')
nltk.download('punkt_tab')
from nltk.tokenize import word_tokenize, sent_tokenize
from nltk.corpus import stopwords
nltk.download('stopwords')
import re
from nltk.stem import WordNetLemmatizer
nltk.download('wordnet')
nltk.download('omw-1.4')
from sklearn.feature_extraction.text import TfidfVectorizer
import numpy as np
from sklearn.metrics.pairwise import cosine_similarity



[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!
[nltk_data] Downloading package stopwords to /root/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


In [ ]:
class PreprocessingModule:
    def __init__(self):
        self.stop_words = set(stopwords.words("english"))
        self.lemmatizer = WordNetLemmatizer()

    def transform(self, text):
        """
        Cleans raw text: tokenizes, removes stopwords/punctuation, lemmatizes.
        Parameters: text (str) - raw input text
        Returns: list of cleaned, lemmatized tokens
        """
        tokens = word_tokenize(text.lower())
        tokens = [self.lemmatizer.lemmatize(token) for token in tokens if token.isalpha() and token not in self.stop_words]
        return tokens


pre = PreprocessingModule()
result = pre.transform("The Dogs are RUNNING quickly!! 😍")
print(result)

['dog', 'running', 'quickly']


In [38]:
class VectorizerModule:
    def __init__(self):
        self.vectorizer = TfidfVectorizer(stop_words='english')
        self.corpus_vectors = None  # will be set after fit()

    def fit(self, corpus):
        """
        Learns vocabulary from the corpus and stores TF-IDF vectors.
        Parameters: corpus (list of str) - the documents to index
        Returns: None
        """
        self.corpus_vectors = self.vectorizer.fit_transform(corpus)

    def transform(self, query):
        """
        Converts a new query into a TF-IDF vector using the already-learned vocabulary.
        Parameters: query (str) - a new piece of text to vectorize
        Returns: sparse vector representation of the query
        """
        query_vector = self.vectorizer.transform([query])
        return query_vector

vec = VectorizerModule()
vec.fit(["I love dogs", "I love cats", "The weather is sunny"])
result = vec.transform("dogs are great")
print(result.shape)


(1, 5)


In [42]:
class Pipeline:
    def __init__(self):
        self.preprocessor = PreprocessingModule()
        self.vectorizer = VectorizerModule()

    def run(self, query, corpus, top_k=3):
        """..."""
        cleaned_corpus = [" ".join(self.preprocessor.transform(doc)) for doc in corpus]
        self.vectorizer.fit(cleaned_corpus)

        cleaned_query = " ".join(self.preprocessor.transform(query))   # ← THIS must come first

        if len(cleaned_query.strip()) == 0:                             # ← THEN this check
               raise ValueError("Query has no valid words after cleaning. Please try a different query.")

        query_vector = self.vectorizer.transform(cleaned_query)
        scores = cosine_similarity(query_vector, self.vectorizer.corpus_vectors)
        top_indices = np.argsort(scores[0])[::-1][:top_k]
        results = [(scores[0][i], corpus[i]) for i in top_indices]
        return results

pipeline = Pipeline()

results = pipeline.run("dogs are great", ["I love dogs", "I love cats", "The weather is sunny"], top_k=2)
for score, sentence in results:
    print(f"{score:.4f} - {sentence}")

0.7960 - I love dogs
0.0000 - The weather is sunny


In [47]:
corpus = [
    # AI
    "Artificial intelligence allows computers to perform tasks that normally require human intelligence.",
    "Machine learning enables systems to learn patterns from data and make predictions.",
    "Natural language processing helps computers understand and process human language.",
    "Neural networks are widely used for image recognition and language processing.",

    # JavaScript
    "JavaScript is a programming language commonly used to build interactive web applications.",
    "Promises in JavaScript are used to handle asynchronous operations.",
    "Arrays allow developers to store and manipulate collections of values.",
    "Functions help organize reusable pieces of JavaScript code.",

    # React
    "React is a JavaScript library for building user interfaces.",
    "React components allow developers to divide an application into reusable UI pieces.",
    "The useState hook allows React components to manage state.",
    "Props are used to pass data from a parent component to a child component.",

    # Cybersecurity
    "Cybersecurity protects computer systems and networks from digital attacks.",
    "Strong passwords and multi-factor authentication improve account security.",
    "A firewall monitors and controls incoming and outgoing network traffic."
]

queries = [
    "How do computers understand human language?",
    "How can I handle asynchronous tasks in JavaScript?",
    "What is used to build reusable user interfaces?",
    "How can I protect a computer network from attacks?",
    "How do systems learn patterns and make predictions?"
]

for q in queries:
    print(f"\n=== Query: {q} ===")
    results = pipeline.run(q, corpus, top_k=3)
    for score, sentence in results:
        print(f"{score:.4f} - {sentence}")


=== Query: How do computers understand human language? ===
0.7195 - Natural language processing helps computers understand and process human language.
0.2364 - Artificial intelligence allows computers to perform tasks that normally require human intelligence.
0.1532 - Cybersecurity protects computer systems and networks from digital attacks.

=== Query: How can I handle asynchronous tasks in JavaScript? ===
0.5974 - Promises in JavaScript are used to handle asynchronous operations.
0.1605 - Artificial intelligence allows computers to perform tasks that normally require human intelligence.
0.1170 - React is a JavaScript library for building user interfaces.

=== Query: What is used to build reusable user interfaces? ===
0.4300 - React is a JavaScript library for building user interfaces.
0.2674 - JavaScript is a programming language commonly used to build interactive web applications.
0.1525 - Functions help organize reusable pieces of JavaScript code.

=== Query: How can I protect a c

In [48]:
edge_cases = ["", "a", "12345 !@#$%"]

for case in edge_cases:
    print(f"\n=== Edge case: '{case}' ===")
    try:
        results = pipeline.run(case, corpus, top_k=3)
        for score, sentence in results:
            print(f"{score:.4f} - {sentence}")
    except ValueError as e:
        print("Caught error:", e)


=== Edge case: '' ===
Caught error: Query has no valid words after cleaning. Please try a different query.

=== Edge case: 'a' ===
Caught error: Query has no valid words after cleaning. Please try a different query.

=== Edge case: '12345 !@#$%' ===
Caught error: Query has no valid words after cleaning. Please try a different query.
